<a href="https://colab.research.google.com/github/velchan15/AI_Data_Process/blob/main/Tugas%202%20-%20Integrasi/Tugas2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pelatihan AI Hari 2 — Integrasi & Instalasi Solusi AI

Notebook ini mengikuti urutan PPT Hari 2. Jalankan cell dari atas ke bawah.

**Tujuan akhir:** membuat model sederhana, menyimpannya sebagai artefak, menghubungkannya ke fungsi prediksi, lalu memasang antarmuka Gradio sementara di Google Colab.

> Gunakan hanya dataset sintetis yang diberikan. Jangan mengunggah data pelanggan perusahaan.


## 📘 Panduan singkat Sesi 2
**Cara pakai:** jalankan sel dari atas ke bawah (**Shift + Enter**), jangan loncat. Di atas tiap sel kode ada kotak **📘 Penjelasan**: apa yang dilakukan, artinya apa, hasil yang benar, dan kalau berbeda harus ngapain.

**Bukti yang harus disimpan:**
| Screenshot | Isi yang harus terlihat |
|---|---|
| 1 | status **Connected** + empat versi library |
| 2 | ukuran data, pembagian latih/uji, akurasi, dua nama file model |
| 3 | tiga tes cepat, form Gradio, hasil prediksi |

**File yang diunduh di akhir:** `model_churn_hari2.joblib`, `metadata_model_hari2.json`, `app_hari2.py`, `requirements_hari2_terpasang.txt`.

## Bagian A — Siapkan Environment

Instalasi di workshop berarti menyiapkan runtime, library, versi, dan bukti. Ini adalah lingkungan pilot, bukan produksi.


### 📘 Penjelasan — Memasang library
**Apa yang dilakukan:** memasang library yang dibutuhkan di Colab.
**Penjelasan:** `!pip install` adalah perintah memasang paket Python (tanda `!` = dijalankan di terminal Colab, bukan kode Python biasa). Library-nya: *scikit-learn* untuk model, *gradio* untuk form web, *joblib* untuk menyimpan model. Tanda `<2` dan `<7` membatasi versi supaya hasilmu sama dengan materi.
**Hasil yang benar:** selesai tanpa tulisan merah. Kalau Colab meminta restart, pilih **Restart session** lalu lanjut.
**Kalau berbeda:** jangan klik instal berulang-ulang. Baca baris terakhir pesan error.

In [1]:
!pip -q install "scikit-learn<2" "gradio<7" joblib


### 📘 Penjelasan — Impor library dan catat versi
**Apa yang dilakukan:** memanggil library lalu menampilkan versinya.
**Penjelasan:** versi dicatat sebagai **bukti lingkungan** — instalasi harus bisa dibuktikan. Versi ini juga akan masuk ke metadata model.
**Hasil yang benar:** muncul lima baris versi (Python, pandas, scikit-learn, joblib, gradio) tanpa error.
📸 **Screenshot 1:** ambil layar yang menampilkan status **Connected** (kanan atas) dan keempat versi library.

In [2]:
import sys
import json
import platform
from pathlib import Path

import pandas as pd
import sklearn
import joblib
import gradio as gr

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("gradio:", gr.__version__)


Python: 3.13.18
pandas: 2.2.3
scikit-learn: 1.6.1
joblib: 1.6.0
gradio: 6.31.0


## Bagian B — Masukkan dataset bersih


### 📘 Penjelasan — Mengunggah dataset
**Apa yang dilakukan:** membuka dialog unggah file, lalu menyimpan nama file ke variabel `DATA_FILE`.
**Penjelasan:** pilih `data_pelanggan_model_hari2.csv` (dataset sintetis 160 baris). Jangan mengunggah data pelanggan nyata.
**Hasil yang benar:** `File: data_pelanggan_model_hari2.csv`
**Kalau berbeda:** nama berakhiran `(1)` atau `_1` berarti file terunggah dua kali. Jalankan ulang dari atas.

In [3]:
from google.colab import files

uploaded = files.upload()
DATA_FILE = next(iter(uploaded))
print("File:", DATA_FILE)


Saving data_pelanggan_model_hari2.csv to data_pelanggan_model_hari2.csv
File: data_pelanggan_model_hari2.csv


### 📘 Penjelasan — Membaca dan memeriksa dataset
**Apa yang dilakukan:** membaca CSV jadi tabel, lalu memeriksa bentuknya.
**Penjelasan:** `assert` itu seperti satpam: kalau nama kolom atau ukuran tabel tidak sesuai, program langsung berhenti dengan pesan, jadi kita tidak melatih model dengan data yang salah. `value_counts()` menghitung berapa pelanggan yang churn.
**Hasil yang benar:** `Ukuran: (160, 7)` (160 baris, 7 kolom). Churn: `No` 96, `Yes` 64.
📸 *Bahan Screenshot 2* (ukuran data).

In [4]:
df = pd.read_csv(DATA_FILE)

expected_columns = [
    "customer_id", "age", "contract", "monthly_charge",
    "tenure_months", "support_calls", "churn"
]
assert df.columns.tolist() == expected_columns, "Nama/urutan kolom berbeda"
assert df.shape == (160, 7), "Ukuran dataset berbeda"

print("Ukuran:", df.shape)
print(df["churn"].value_counts())
df.head()


Ukuran: (160, 7)
churn
No     96
Yes    64
Name: count, dtype: int64


,customer_id,age,contract,monthly_charge,tenure_months,support_calls,churn
0,C0001,36,Annual,160000,52,6,No
1,C0002,47,Monthly,190000,22,4,Yes
2,C0003,53,Monthly,160000,53,1,No
3,C0004,31,Annual,160000,28,4,Yes
4,C0005,32,Monthly,180000,15,1,Yes


## Bagian C — Bangun komponen model

Kita memakai Pipeline agar preprocessing dan model selalu berjalan dalam urutan yang sama.


### 📘 Penjelasan — Memisahkan input (X) dan target (y)
**Apa yang dilakukan:** memisahkan "soal" dan "kunci jawaban".
**Penjelasan:** **X** = 5 kolom ciri pelanggan (umur, kontrak, tagihan, lama berlangganan, panggilan dukungan). `customer_id` **tidak ikut** karena hanya penanda, bukan ciri yang menentukan churn. **y** = kolom `churn` diubah jadi angka (`No` → 0, `Yes` → 1) karena model hanya mengerti angka.
**Hasil yang benar:** `Input: (160, 5)` dan `Target: (160,)`.
**Kalau berbeda:** `customer_id` ikut jadi fitur adalah kesalahan umum, cek daftar `FEATURES`.

In [5]:
from sklearn.model_selection import train_test_split

FEATURES = [
    "age", "contract", "monthly_charge",
    "tenure_months", "support_calls"
]
TARGET = "churn"

X = df[FEATURES].copy()
y = df[TARGET].map({"No": 0, "Yes": 1})

print("Input:", X.shape)
print("Target:", y.shape)


Input: (160, 5)
Target: (160,)


### 📘 Penjelasan — Membagi data latih dan data uji
**Apa yang dilakukan:** membagi data jadi 80% untuk belajar dan 20% untuk ujian.
**Penjelasan:** seperti belajar dulu lalu ujian, soal ujian tidak boleh dilihat saat belajar. `random_state=42` membuat pembagian selalu sama. `stratify=y` menjaga perbandingan Yes/No tetap seimbang di kedua bagian.
**Hasil yang benar:** `Train: (128, 5)` dan `Test: (32, 5)`.
📸 *Bahan Screenshot 2* (pembagian data).

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)


Train: (128, 5)
Test: (32, 5)


### 📘 Penjelasan — Menyusun pipeline
**Apa yang dilakukan:** merakit "resep tetap" yang berisi dua tahap.
**Penjelasan:** tahap 1 **preprocessing**: angka distandarkan (`StandardScaler`), kolom kontrak diubah jadi kolom 0/1 (`OneHotEncoder`). Tahap 2 **model** `LogisticRegression` yang menghitung peluang churn. Keduanya dibungkus satu `Pipeline` agar urutannya tidak pernah tertukar dan bisa disimpan jadi satu file.
**Hasil yang benar:** `Pipeline siap` (modelnya belum dilatih).

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

numeric_features = [
    "age", "monthly_charge", "tenure_months", "support_calls"
]
categorical_features = ["contract"]

preprocess = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("category", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

print("Pipeline siap")


Pipeline siap


### 📘 Penjelasan — Melatih dan menguji model
**Apa yang dilakukan:** melatih model, lalu mengujinya pada data uji.
**Penjelasan:** `fit` = belajar dari data latih. `predict` = menebak data uji. `accuracy_score` = persentase tebakan yang benar. Akurasi ini hanya **pemeriksaan fungsi**, bukan persetujuan produksi.
**Hasil yang benar:** `Akurasi uji: 0.781` (sekitar 0,78 = 25 dari 32 tebakan benar).
📸 *Bahan Screenshot 2* (akurasi).

In [8]:
from sklearn.metrics import accuracy_score

pipeline.fit(X_train, y_train)
pred_test = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, pred_test)

print(f"Akurasi uji: {accuracy:.3f}")
print("Catatan: metrik ini hanya pemeriksaan fungsi, bukan persetujuan produksi.")


Akurasi uji: 0.781
Catatan: metrik ini hanya pemeriksaan fungsi, bukan persetujuan produksi.


## Bagian D — Simpan artefak dan metadata

Jangan pernah memuat file model joblib/pickle dari sumber yang tidak dipercaya.


### 📘 Penjelasan — Menyimpan model dan metadata
**Apa yang dilakukan:** menyimpan model jadi satu file, ditambah "kartu identitas" model.
**Penjelasan:** `joblib.dump` menyimpan pipeline ke file `.joblib`. **Metadata JSON** mencatat nama, versi, fitur, jumlah data, akurasi, dan versi library, supaya model bisa dilacak. Di Sesi 3 versi ini akan dicocokkan sebelum model dibuka. **Jangan pernah membuka file joblib dari sumber yang tidak dipercaya.**
**Hasil yang benar:** dua baris `Tersimpan:` (model dan metadata).
📸 **Screenshot 2:** ukuran data, pembagian, akurasi, dan dua nama file. Scroll supaya hasil sel-sel sebelumnya ikut terlihat.

In [9]:
MODEL_FILE = "model_churn_hari2.joblib"
METADATA_FILE = "metadata_model_hari2.json"

joblib.dump(pipeline, MODEL_FILE)

metadata = {
    "model_name": "demo_churn_hari2",
    "model_version": "1.0-workshop",
    "features": FEATURES,
    "target": TARGET,
    "threshold": 0.50,
    "training_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "test_accuracy": round(float(accuracy), 4),
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
    "joblib_version": joblib.__version__,
}

Path(METADATA_FILE).write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)

print("Tersimpan:", MODEL_FILE)
print("Tersimpan:", METADATA_FILE)


Tersimpan: model_churn_hari2.joblib
Tersimpan: metadata_model_hari2.json


## Bagian E — Integrasikan model dengan fungsi prediksi


### 📘 Penjelasan — Membuka kembali model
**Apa yang dilakukan:** memuat model dari file yang baru disimpan.
**Penjelasan:** ini membuktikan file model benar-benar bisa dipakai, bukan sekadar model yang masih ada di memori.
**Hasil yang benar:** `Model berhasil dimuat`.

In [10]:
model = joblib.load(MODEL_FILE)
print("Model berhasil dimuat")


Model berhasil dimuat


### 📘 Penjelasan — Fungsi prediksi (adapter)
**Apa yang dilakukan:** membuat fungsi yang mengubah 5 input pengguna menjadi hasil yang mudah dibaca.
**Penjelasan:** fungsi ini seperti colokan universal: input dari form dijadikan tabel satu baris, model menghitung peluang churn (`predict_proba`), lalu kalau peluang ≥ 50% hasilnya **RISIKO TINGGI**, selain itu **RISIKO RENDAH**. Nama dan urutan 5 input harus sama dengan saat model dilatih. Inilah **kontrak input**.
**Hasil yang benar:** tidak ada output, karena hanya mendefinisikan fungsi.

In [11]:
def predict_churn(age, contract, monthly_charge, tenure_months, support_calls):
    row = pd.DataFrame([{
        "age": float(age),
        "contract": str(contract),
        "monthly_charge": float(monthly_charge),
        "tenure_months": float(tenure_months),
        "support_calls": float(support_calls),
    }])

    probability = float(model.predict_proba(row)[0, 1])
    label = "RISIKO TINGGI" if probability >= 0.50 else "RISIKO RENDAH"
    return f"{label} | peluang churn: {probability:.1%}"


### 📘 Penjelasan — Tiga tes cepat (smoke test)
**Apa yang dilakukan:** mencoba fungsi dengan tiga contoh pelanggan.
**Penjelasan:** tes cepat memastikan fungsi menjawab masuk akal sebelum dipasang ke form. Pelanggan kontrak bulanan dengan tagihan besar dan banyak panggilan dukungan wajar berisiko tinggi; pelanggan kontrak tahunan yang lama berlangganan wajar berisiko rendah.
**Hasil yang benar (sekitar):** TINGGI ± 98%, RENDAH ± 1%, TINGGI ± 88%. Angkanya boleh beda sedikit, tapi **labelnya harus sama**.
📸 *Bahan Screenshot 3* (tiga tes cepat).

In [12]:
smoke_tests = [
    (35, "Monthly", 180000, 4, 6),
    (50, "Annual", 80000, 48, 0),
    (25, "Monthly", 120000, 6, 3),
]

for test in smoke_tests:
    print(test, "->", predict_churn(*test))


(35, 'Monthly', 180000, 4, 6) -> RISIKO TINGGI | peluang churn: 98.5%
(50, 'Annual', 80000, 48, 0) -> RISIKO RENDAH | peluang churn: 0.6%
(25, 'Monthly', 120000, 6, 3) -> RISIKO TINGGI | peluang churn: 88.0%


## Bagian F — Pasang antarmuka Gradio


### 📘 Penjelasan — Merakit antarmuka Gradio
**Apa yang dilakukan:** membuat form web dari fungsi prediksi.
**Penjelasan:** lima kontrol (slider, pilihan, angka) menjadi lima input fungsi, dan satu kotak teks menampilkan hasil. Di sini form baru **dirakit**, belum tampil.
**Hasil yang benar:** `Interface siap`.

In [13]:
demo = gr.Interface(
    fn=predict_churn,
    inputs=[
        gr.Slider(18, 100, value=35, step=1, label="Umur"),
        gr.Radio(["Monthly", "Annual"], value="Monthly", label="Kontrak"),
        gr.Number(value=120000, label="Tagihan bulanan"),
        gr.Slider(1, 72, value=12, step=1, label="Lama berlangganan (bulan)"),
        gr.Slider(0, 10, value=2, step=1, label="Jumlah panggilan dukungan"),
    ],
    outputs=gr.Textbox(label="Hasil prediksi"),
    title="Demo Prediksi Churn",
    description="Data sintetis untuk latihan. Hasil bukan keputusan bisnis.",
)

print("Interface siap")


Interface siap


### 📘 Penjelasan — Menjalankan demo sementara
**Apa yang dilakukan:** menampilkan form dan membuat tautan publik sementara.
**Penjelasan:** `share=True` membuat tautan yang **bisa dibuka siapa saja** selama runtime hidup. Karena itu jangan mengetik data asli (NIK, email) di form. Coba isi form lalu klik **Submit**.
**Hasil yang benar:** muncul form dan tautan `https://....gradio.live`, dan hasil prediksi tampil setelah Submit.
📸 **Screenshot 3:** tiga tes cepat, form, dan hasil prediksi.

In [14]:
# Tautan share bersifat sementara dan dapat diakses publik.
# Jangan masukkan data nyata.
demo.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e849f82fd4a26cd9b9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Bagian G — Simpan bukti instalasi


### 📘 Penjelasan — Menyimpan bukti instalasi
**Apa yang dilakukan:** menyimpan dua file bukti.
**Penjelasan:** `app_hari2.py` adalah kode demo yang bisa berdiri sendiri. `requirements_hari2_terpasang.txt` mencatat **versi persis** library yang terpasang, jadi orang lain bisa memasang lingkungan yang sama (handoff).
**Hasil yang benar:** dua baris `Tersimpan:`.

In [15]:
APP_FILE = "app_hari2.py"
REQUIREMENTS_FILE = "requirements_hari2_terpasang.txt"

app_code = "import joblib\nimport pandas as pd\nimport gradio as gr\n\nMODEL_FILE = \"model_churn_hari2.joblib\"\nmodel = joblib.load(MODEL_FILE)\n\ndef predict_churn(age, contract, monthly_charge, tenure_months, support_calls):\n    row = pd.DataFrame([{\n        \"age\": float(age),\n        \"contract\": str(contract),\n        \"monthly_charge\": float(monthly_charge),\n        \"tenure_months\": float(tenure_months),\n        \"support_calls\": float(support_calls),\n    }])\n    probability = float(model.predict_proba(row)[0, 1])\n    label = \"RISIKO TINGGI\" if probability >= 0.50 else \"RISIKO RENDAH\"\n    return f\"{label} | peluang churn: {probability:.1%}\"\n\ndemo = gr.Interface(\n    fn=predict_churn,\n    inputs=[\n        gr.Slider(18, 100, value=35, step=1, label=\"Umur\"),\n        gr.Radio([\"Monthly\", \"Annual\"], value=\"Monthly\", label=\"Kontrak\"),\n        gr.Number(value=120000, label=\"Tagihan bulanan\"),\n        gr.Slider(1, 72, value=12, step=1, label=\"Lama berlangganan (bulan)\"),\n        gr.Slider(0, 10, value=2, step=1, label=\"Jumlah panggilan dukungan\"),\n    ],\n    outputs=gr.Textbox(label=\"Hasil prediksi\"),\n    title=\"Demo Prediksi Churn\",\n    description=\"Data sintetis untuk latihan. Hasil bukan keputusan bisnis.\",\n)\n\nif __name__ == \"__main__\":\n    demo.launch()\n"
Path(APP_FILE).write_text(app_code, encoding="utf-8")

requirements = "\n".join([
    f"pandas=={pd.__version__}",
    f"scikit-learn=={sklearn.__version__}",
    f"joblib=={joblib.__version__}",
    f"gradio=={gr.__version__}",
]) + "\n"
Path(REQUIREMENTS_FILE).write_text(requirements, encoding="utf-8")

print("Tersimpan:", APP_FILE)
print("Tersimpan:", REQUIREMENTS_FILE)


Tersimpan: app_hari2.py
Tersimpan: requirements_hari2_terpasang.txt


### 📘 Penjelasan — Mengunduh hasil akhir
**Apa yang dilakukan:** mengunduh keempat file hasil ke komputermu.
**Penjelasan:** kalau browser menanyakan izin unduh banyak file, pilih **Allow**. Simpan di folder Sesi 2 bersama notebook dan screenshot.
**Hasil yang benar:** empat file terunduh. Jangan lupa tutup runtime setelah selesai.

In [16]:
from google.colab import files

output_files = [
    MODEL_FILE,
    METADATA_FILE,
    APP_FILE,
    REQUIREMENTS_FILE,
]

for name in output_files:
    files.download(name)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Selesai

Simpan notebook dan empat artefak hasil. Pada Hari 3, artefak ini akan dipakai untuk latihan perawatan, monitoring, dan penanganan perubahan.
